In [ ]:
import pandas as pd
import os
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import numpy as np
import seaborn as sns
import shutil


In [11]:
df = pd.read_parquet('image_metadata.parquet')
print(f"base: {len(df)} images")
df = df[~df['filename'].str.startswith(('[DUPE]', '[LQ]'))]
print(f"filtered: {len(df)} images")

base: 2341506 images
filtered: 2223389 images


In [12]:
TARGET_SIZE = 1000
SEED = 42

df_sample = df.groupby(['category', 'model_type']).sample(n=TARGET_SIZE, random_state=SEED)
print(df.groupby(['category', 'model_type']).size())
df_sample.to_csv('team_subset_metadata.csv', index=False)
len(df)

category  model_type           
ai        ADM                      167994
          Glide                    167999
          Midjourney               167985
          Stable Diffusion v1.4    167997
          Stable Diffusion v1.5    173854
          VQDM                     168000
          Wukong                   168000
nature    ADM                      148075
          Glide                    150737
          Midjourney               150132
          Stable Diffusion v1.4    150186
          Stable Diffusion v1.5    143623
          VQDM                     149816
          Wukong                   148991
dtype: int64


2223389

In [14]:
df_sample.head()

,path,filename,model_type,category,width,height,aspect_ratio,megapixels,size_bytes,format
2099097,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai...,61_adm_48.PNG,ADM,ai,256,256,1.0,0.07,156526,PNG
2019555,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai...,178_adm_48.PNG,ADM,ai,256,256,1.0,0.07,148347,PNG
2062709,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai...,418_adm_114.PNG,ADM,ai,256,256,1.0,0.07,62364,PNG
2140330,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai...,84_adm_137.PNG,ADM,ai,256,256,1.0,0.07,188952,PNG
2064594,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai...,428_adm_50.PNG,ADM,ai,256,256,1.0,0.07,171872,PNG


In [16]:
NAME_MAP = {
    'adm': 'ADM',
    'glide': 'Glide',
    'midjourney': 'Midjourney',
    'sdv4': 'Stable Diffusion v1.4',
    'sdv5': 'Stable Diffusion v1.5',
    'vqdm': 'VQDM',
    'wukong': 'Wukong'
}
REVERSE_MAP = {v: k for k, v in NAME_MAP.items()}

EXPORT_PATH = Path('../data/subset').resolve()
EXPORT_PATH.mkdir(exist_ok=True)

# Create subdirectories to keep things organized
(EXPORT_PATH / 'ai').mkdir(exist_ok=True)
(EXPORT_PATH / 'nature').mkdir(exist_ok=True)

print(f"Copying files to {EXPORT_PATH}...")

copy_errors = 0

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Copying"):
    source_path = Path(row['path'])
    category = row['category']
    model_type = row['model_type']
    model_folder = REVERSE_MAP.get(model_type)
    # Define destination, make parent folders if needed.
    dest_dir = EXPORT_PATH / category / model_folder
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / source_path.name
    
    try:
        shutil.copy2(source_path, dest_path) # copy2 preserves metadata like timestamps
    except FileNotFoundError:
        copy_errors += 1

print(f"\nDone! Files copied to: {EXPORT_PATH}")
if copy_errors > 0:
    print(f"Warning: {copy_errors} files were not found and skipped.")

Copying files to /workspace/AML-3-MVL-AI-Classifier/data/subset...


Copying: 100%|██████████| 14000/14000 [00:11<00:00, 1257.00it/s]


Done! Files copied to: /workspace/AML-3-MVL-AI-Classifier/data/subset
